# 05 Why do some queries fail under every model?

This notebook is self-contained. It downloads NFCorpus and four embedding models from the Hugging Face Hub, encodes all text, and calculates all numbers and figures. It does not read `results/`, `cache/`, or a `silent_rag` module. You need only the pip packages in the next cell. On a laptop CPU, the run time is approximately 15 to 30 minutes. The notebook encodes the corpus one time for each model.

**Questions**
1. Do the models share failures more than chance?
2. What are the properties of universal failures? Does the data support the "indirect judgment" explanation when the number of judged documents is a control?
3. Does the failure rate change with the split?
4. How many failures does a different model fix, and how many new failures does it cause?

**Method**
- Observed against expected failure counts under independence (Poisson-binomial, Monte Carlo). This is only a reference scale.
- Pairwise Cohen's kappa.
- A graded-relevance comparison with Fisher and Mann-Whitney tests and Holm correction.
- A logistic regression that adjusts the level-1 comparison for `log(n_rel)`.
- Wilson intervals for each split.
- Rescue and break counts with exact McNemar tests.

The notebook also makes two separate blind rating sheets for a human audit. **No human ratings exist. Thus, this notebook makes no claim about the results of the raters.** Each inferential section has a "Why this test" note. The note gives the estimand, the unit of independence, the assumptions, and the Monte Carlo resolution.

**Scope.** This notebook uses only NFCorpus. NFCorpus is the development set of the project. SciFact and FiQA are held-out sets for the final run. This notebook does not load them.

A query "fails" under a model when none of its relevant documents is in the top 10 (recall@10 = 0). A **universal failure** fails under all four models. A **control** fails under no model.

In [ ]:
# On a new environment, remove the "#" from the next line.
# %pip install -q datasets==5.1.0 sentence-transformers==6.1.0 torch==2.14.1 scikit-learn==1.9.1 scipy==1.18.1 pandas==3.0.6 numpy==2.5.4 matplotlib

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # On macOS, this prevents an OpenMP conflict. It has no effect on other systems.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
DEPTH = 100
SPLITS = ["train", "validation", "test"]   # The Hub name of the qrels split is "validation", not "dev".
MAX_SEQ = 512
SAVE_AUDIT_SHEET = False  # Set to True to write the two audit sheets and their keys to audit_out/. Do not commit them: the keys contain qrel data.
AUDIT_DIR = "audit_out"

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODELS = {
    "multi-qa-MiniLM": dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0", query_prefix="", doc_prefix=""),
    "all-MiniLM": dict(repo="sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", query_prefix="", doc_prefix=""),
    "bge-small": dict(repo="BAAI/bge-small-en-v1.5", revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a", query_prefix="Represent this sentence for searching relevant passages: ", doc_prefix=""),
    "e5-small": dict(repo="intfloat/e5-small-v2", revision="ffb93f3bd4047442299a41ebb6fa998a38507c52", query_prefix="query: ", doc_prefix="passage: "),
}
labels = list(MODELS)
BASELINE = labels[0]

## Helpers, data, and clean retrieval for all four models

This section defines the functions that load the data, encode the text, do an exact cosine search, and calculate recall@10 (no nDCG). The notebook uses all labelled queries that have one or more relevant documents, from the train, validation, and test qrels. If a query is in more than one split, the notebook keeps the first split. `first_rank` is the rank of the first relevant document. A value of 101 means that the document is not in the top 100.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def search(D, Q, depth=DEPTH):
    S = Q @ D.T
    # Stable full sort: if two scores are equal (the corpus has duplicate documents), the lower document index comes first.
    # Thus, the top-k does not depend on the search routine.
    return np.argsort(-S, axis=1, kind="stable")[:, :depth]


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
corpus = dict(zip(doc_ids, docs))
pos = {d: i for i, d in enumerate(doc_ids)}
qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]
split = np.array([split_of[q] for q in qids])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]

runs = {}
for m in labels:
    spec = MODELS[m]
    enc = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
    enc.max_seq_length = MAX_SEQ
    D = encode(enc, [spec["doc_prefix"] + t for t in docs])
    Q = encode(enc, [spec["query_prefix"] + queries[q] for q in qids])
    top = search(D, Q)
    ids = [[doc_ids[j] for j in row] for row in top]
    recall = np.array([len(r & set(i[:K])) / len(r) for r, i in zip(rel, ids)])
    first = np.array([min([j + 1 for j, d in enumerate(i) if d in r], default=DEPTH + 1) for r, i in zip(rel, ids)])
    runs[m] = {"enc": enc, "D": D, "Q": Q, "ids": ids, "recall": recall, "first_rank": first}
    print(m, "failing (recall@10 = 0):", int((recall == 0).sum()), "of", len(qids), flush=True)

F = np.column_stack([runs[m]["recall"] == 0 for m in labels])   # True: the query failed
n_fail = F.sum(1)
U, C = n_fail == 4, n_fail == 0
print(len(qids), "queries | failing under all four:", int(U.sum()), "| under none:", int(C.sum()))

## 1. Do the models share failures more than chance?

Under independence, each query fails under model `m` with the marginal probability `p_m` of that model. Thus, the number of models that fail has a Poisson-binomial distribution. The table compares the expected and observed number of queries for each count (0 to 4). It also gives a Monte Carlo p-value for the observed number of universal failures, from 5,000 simulated datasets. The simulation runs in chunks of 500 to keep memory use small. The pairwise kappa between the failure indicators of the models has a bootstrap CI over queries.

**Why this test**
- **Estimand:** how many more universal failures occur than four models that fail *independently at their marginal rates* give.
- **Unit of independence:** the query (simulation and kappa bootstrap).
- **Assumptions:** the null assumes that all queries have the same failure probability under a model. This is false: the difficulty of queries changes very much (1 to 1,363 relevant documents). **Thus, the null is a straw man, and a rejection is expected.** The notebook keeps it only as a reference scale. This notebook does not calculate a difficulty-adjusted null. The logistic adjustment in section 2 is a partial answer.
- **Why not the usual alternative:** a chi-square test on the 0 to 4 count table tests the same straw-man null with an asymptotic approximation. Kappa is shown because it corrects pairwise agreement for the marginal rates.
- **Resolution / Monte Carlo error:** the lowest possible p is `1/5001 ≈ 0.0002`, reported as `p < 0.001`. The kappa CI comes from 500 query-bootstrap replicates. It ignores topic clustering.

In [ ]:
p = F.mean(0)
rng = np.random.default_rng(SEED)
N_SIM, CHUNK = 5000, 500
count_hist = np.zeros(5)       # Sum over simulations: number of queries with k failing models
exp_all = np.empty(N_SIM, dtype=int)
for start in range(0, N_SIM, CHUNK):
    sim = (rng.random((CHUNK, len(qids), 4)) < p).sum(2)     # Chunks: approximately 50 MB for each chunk, not 520 MB in one array
    count_hist += np.array([(sim == k).sum() for k in range(5)])
    exp_all[start:start + CHUNK] = (sim == 4).sum(1)
expected = count_hist / N_SIM
observed = np.array([(n_fail == k).sum() for k in range(5)])
dep = pd.DataFrame({"models_failing": range(5), "observed": observed, "expected_if_independent": expected})
dep["ratio"] = dep["observed"] / dep["expected_if_independent"]
print("Monte-Carlo p for observed universal failures under independence:", (1 + (exp_all >= observed[4]).sum()) / (N_SIM + 1))
display(dep.round(2))

rows = []
for i in range(4):
    for j in range(i + 1, 4):
        k = cohen_kappa_score(F[:, i], F[:, j])
        boot = [cohen_kappa_score(F[s, i], F[s, j]) for s in (rng.integers(0, len(qids), len(qids)) for _ in range(500))]
        rows.append({"pair": f"{labels[i]} vs {labels[j]}", "kappa": k, "ci_lo": np.percentile(boot, 2.5), "ci_hi": np.percentile(boot, 97.5)})
pd.DataFrame(rows).round(3)

## 2. Relevance level, and the properties of the failures

For each query, this section uses three values:
- the highest relevance level of its judgments (NFCorpus uses 2 = directly cited, 1 = indirectly linked)
- the number of relevant documents
- the query length in tokens

If most universal failures are queries with a best judgment of level 1, this supports the "indirect judgment" explanation. The tests are Fisher exact for "best judgment is level 1", Mann-Whitney with the rank-biserial effect size for the other values, and bootstrap CIs for the differences. The p-values are Holm-adjusted across the **three** tests in the first table.

**Manifestations are separate from explanations.** Two values restate the outcome: the rank of the first relevant document, and the cosine between the top-1 document of the baseline and the nearest relevant document. By definition, a failing query has its first relevant document after rank 10. When the top-1 document is relevant, the cosine is exactly 1.0 (this is frequent among controls). Thus, these values are in a separate descriptive table, with no test, and they are not in the Holm family.

**Why this test**
- **Estimand:** the differences in pre-outcome query properties between universal failures and controls (queries that fail under no model).
- **Unit of independence:** the query.
- **Assumptions:** Fisher's exact test conditions on the margins and assumes independent queries. Mann-Whitney tests stochastic dominance, not a difference in means. Thus, the table gives the rank-biserial effect size with it. Both tests ignore topic clustering, so the p-values are optimistic.
- **Why not the usual alternative:** a t-test on `n_rel` (range 1 to 1,363, very skewed) depends mostly on a few queries. A chi-square test on a 2×2 table with one small cell is less exact than Fisher. Holm has the same guarantee as Bonferroni and more power.
- **Resolution / Monte Carlo error:** exact or asymptotic p-values, with no Monte Carlo limit. The bootstrap CIs use 2,000 replicates.

In [ ]:
base = runs[BASELINE]
max_level = np.array([max(qrels[q].values()) for q in qids])
n_rel = np.array([len(r) for r in rel])
first_rank = base["first_rank"]
D, Q = base["D"], base["Q"]
near_miss = np.array([float(max(D[pos[d]] @ D[pos[base["ids"][i][0]]] for d in rel[i] if d in pos)) for i in range(len(qids))])
top1_is_relevant = np.array([base["ids"][i][0] in rel[i] for i in range(len(qids))])
q_tokens = np.array([len(base["enc"].tokenizer.tokenize(queries[q])) for q in qids])


def rank_biserial(a, b):
    return 2 * stats.mannwhitneyu(a, b).statistic / (len(a) * len(b)) - 1


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out, run = np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (len(p) - rank)))
        out[i] = run
    return out


rows = []
level1_u, level1_c = max_level[U] == 1, max_level[C] == 1
odds, p_f = stats.fisher_exact([[level1_u.sum(), (~level1_u).sum()], [level1_c.sum(), (~level1_c).sum()]])
rng = np.random.default_rng(SEED)
bd = [level1_u[rng.integers(0, U.sum(), U.sum())].mean() - level1_c[rng.integers(0, C.sum(), C.sum())].mean() for _ in range(2000)]
rows.append({"measure": "share with best judgment = level 1", "universal_failures": level1_u.mean(), "controls": level1_c.mean(),
             "diff": level1_u.mean() - level1_c.mean(), "ci_lo": np.percentile(bd, 2.5), "ci_hi": np.percentile(bd, 97.5),
             "effect_rank_biserial": np.nan, "test": "Fisher exact", "p": p_f})
for name_, arr in (("number of relevant docs", n_rel), ("query tokens", q_tokens)):
    a, b = arr[U].astype(float), arr[C].astype(float)
    bd = [a[rng.integers(0, len(a), len(a))].mean() - b[rng.integers(0, len(b), len(b))].mean() for _ in range(2000)]
    rows.append({"measure": name_ + " (mean)", "universal_failures": a.mean(), "controls": b.mean(), "diff": a.mean() - b.mean(),
                 "ci_lo": np.percentile(bd, 2.5), "ci_hi": np.percentile(bd, 97.5), "effect_rank_biserial": rank_biserial(a, b),
                 "test": "Mann-Whitney", "p": stats.mannwhitneyu(a, b).pvalue})
chars = pd.DataFrame(rows)
chars["p_holm"] = holm(chars["p"])
chars["n_universal"], chars["n_controls"] = int(U.sum()), int(C.sum())
display(chars.round(4))

# Manifestations of failure: descriptive only, with no test. They restate the outcome.
man = []
for name_, arr in (("rank of first relevant (baseline; 101 = beyond top 100)", first_rank), ("cosine: top-1 to nearest relevant", near_miss)):
    man.append({"manifestation": name_, "universal_failures_mean": arr[U].mean(), "controls_mean": arr[C].mean(),
                "universal_failures_median": np.median(arr[U]), "controls_median": np.median(arr[C])})
print("Manifestations (restate the definition of failure; not tested):")
display(pd.DataFrame(man).round(3))
print("share of queries whose baseline top-1 document is itself relevant (cosine = 1.0 by construction): controls",
      round(float(top1_is_relevant[C].mean()), 3), "| universal failures", round(float(top1_is_relevant[U].mean()), 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
ax = axes[0]
ax.hist([np.minimum(first_rank[U], 101), np.minimum(first_rank[C], 101)], bins=np.arange(1, 105, 5), label=["universal failures", "controls"], density=True)
ax.set_xlabel("rank of first relevant (101 = beyond top 100)")
ax.legend(fontsize=7)
axes[1].hist([near_miss[U], near_miss[C]], bins=20, density=True)
axes[1].set_xlabel("cos(top-1, nearest relevant)")
axes[2].bar(["level 1 only", "has level 2"], [level1_u.mean(), 1 - level1_u.mean()], alpha=0.7, label="universal")
axes[2].bar(["level 1 only", "has level 2"], [level1_c.mean(), 1 - level1_c.mean()], alpha=0.5, label="controls")
axes[2].legend(fontsize=7)
fig.tight_layout()
plt.show()

### Does the number of relevant documents explain the level-1 difference?

Queries with few judged documents fail more easily: recall@10 = 0 is more probable with one relevant document than with fifty. These queries can also have only level-1 judgments more frequently. A logistic regression of universal failure on `best judgment = level 1`, with and without `log(n_rel)`, shows whether the level-1 association stays after this adjustment. The notebook fits it on universal failures against controls (as in the previous table). As a sensitivity check, it also fits it on universal failures against all other queries. The odds ratios have 95% query-bootstrap CIs (1,000 replicates).

**Why this test**
- **Estimand:** the odds ratio of universal failure for level-1-only queries, with `log(n_rel)` held constant.
- **Unit of independence:** the query (bootstrap). The CIs ignore topic clustering, so they are too narrow.
- **Assumptions:** universal failure is linear in `log(n_rel)` on the log-odds scale. A comparison of the adjusted and unadjusted coefficients shows whether the adjustment is important.
- **Why not the usual alternative:** strata on `n_rel` give strata with too few queries (the level-1 share is 91% to 97%). The unadjusted Fisher test cannot separate the two explanations.
- **Resolution / Monte Carlo error:** bootstrap percentile CI. The notebook skips and counts the replicates in which a group has no variation.

In [ ]:
level1 = (max_level == 1).astype(float)
log_nrel = np.log(n_rel)


def logit_coefs(X, yb):
    return LogisticRegression(penalty=None, max_iter=5000).fit(X, yb).coef_[0]


rows = []
for pop_name, mask in (("universal vs controls", U | C), ("universal vs all other queries", np.ones(len(qids), bool))):
    yb = U[mask].astype(int)
    for model_name, X in (("level-1 only", level1[mask, None]), ("level-1 + log(n_rel)", np.column_stack([level1[mask], log_nrel[mask]]))):
        est = logit_coefs(X, yb)
        rng = np.random.default_rng(SEED)
        boots, skipped = [], 0
        for _ in range(1000):
            i = rng.integers(0, len(yb), len(yb))
            if yb[i].min() == yb[i].max() or np.ptp(X[i, 0]) == 0:
                skipped += 1
                continue
            boots.append(logit_coefs(X[i], yb[i]))
        boots = np.array(boots)
        for j, term in enumerate(["level-1", "log(n_rel)"][: X.shape[1]]):
            lo, hi = np.percentile(boots[:, j], [2.5, 97.5])
            rows.append({"population": pop_name, "model": model_name, "term": term, "n": len(yb), "coef": est[j],
                         "odds_ratio": np.exp(est[j]), "or_ci_lo": np.exp(lo), "or_ci_hi": np.exp(hi), "boot_skipped": skipped})
pd.DataFrame(rows).round(3)

## 3. Does the failure rate depend on the split?

If the universal-failure rate is different across train, validation, and test, a test-only value does not describe the full dataset. This section gives Wilson 95% intervals for each split and a chi-square test of homogeneity.

**Why this test**
- **Estimand:** the universal-failure rate in each official split, and whether the rates are different.
- **Unit of independence:** the query.
- **Assumptions:** binomial sampling in a split. The chi-square approximation needs expected counts ≥ 5. All cells here meet this (the smallest split has 323 queries, and the rate is approximately 20%).
- **Why not the usual alternative:** the Wald interval has coverage that is too low for proportions. The Wilson interval stays near the nominal coverage. Fisher's exact test on a 3×2 table is not necessary at these counts.
- **Resolution / Monte Carlo error:** analytic, with no Monte Carlo. A result that is not significant here is not an equivalence claim.

In [ ]:
rows = []
for s in SPLITS:
    m = split == s
    k, n = int(U[m].sum()), int(m.sum())
    ci = stats.binomtest(k, n).proportion_ci(method="wilson")
    rows.append({"split": s, "n": n, "universal_failures": k, "rate": k / n, "wilson_lo": ci.low, "wilson_hi": ci.high})
chi = stats.chi2_contingency(np.array([[r["universal_failures"], r["n"] - r["universal_failures"]] for r in rows]))
print("chi-square homogeneity across splits: chi2 =", round(chi[0], 2), "p =", round(chi[1], 4))
pd.DataFrame(rows).round(4)

## 4. Rescue by other models

How many failures of the baseline does each other model fix (rescued)? How many queries that the baseline answers does it newly fail (broken)? The table shows this for all labelled queries and for the 323 test queries. It includes an exact McNemar test of "rescued = broken" on the paired failure indicators.

**Why this test**
- **Estimand:** whether a change from the baseline to a model changes the failure rate. The test uses only the discordant queries (rescued against broken).
- **Unit of independence:** the query (paired across the two models).
- **Assumptions:** under the null, a discordant query is equally probable to be rescued or broken. Thus, rescued ~ Binomial(rescued + broken, 0.5). The test treats queries as independent and ignores topic clustering.
- **Why not the usual alternative:** an unpaired test of the two failure rates (chi-square on two proportions) ignores that both models answered the same queries. It does not use the pairing that gives McNemar its power. The exact binomial version does not need a continuity correction.
- **Resolution / Monte Carlo error:** exact, with no Monte Carlo. The three contrasts in each scope are Holm-adjusted.

In [ ]:
rows = []
for scope, mask in (("all queries", np.ones(len(qids), bool)), ("test queries", split == "test")):
    fb = F[:, 0] & mask
    for j, l in enumerate(labels[1:], 1):
        rescued = int((fb & ~F[:, j]).sum())
        broken = int((~F[:, 0] & F[:, j] & mask).sum())
        rows.append({"scope": scope, "n_queries": int(mask.sum()), "model": l, "baseline_failures": int(fb.sum()),
                     "rescued_by_model": rescued, "newly_failing": broken,
                     "mcnemar_exact_p": stats.binomtest(rescued, rescued + broken, 0.5).pvalue})
resc = pd.DataFrame(rows)
resc["p_holm_within_scope"] = resc.groupby("scope")["mcnemar_exact_p"].transform(lambda s: holm(s.to_numpy()))
resc.round(4)

## 5. Blind audit sheets

**Why two sheets.** A single sheet that shows the retrieved top 3 of a query next to its judged-relevant documents is not blind. For a universal failure, no retrieved document can be judged-relevant. For a control, a retrieved snippet frequently appears again word for word as a judged document. Thus, a rater can find the group from the text alone. The cell after the sheet code counts how frequently this occurs. For this reason, the audit has two sheets:

- **Sheet A (retrieval):** the query text and the top 3 retrieved documents of the baseline. Rated column: `top_doc_answers_query` (0/1).
- **Sheet B (judgments):** the query text and two judged-relevant documents. Rated column: `qrel_doc_plausible` (0/1/2).

Each sheet has its own independently shuffled blind IDs and its own key. Thus, row order and IDs cannot link the two sheets. Sheet B shows only judged-relevant documents **outside the top 10 of the baseline** whose text does not match a retrieved snippet (the corpus has duplicate documents). Thus, by construction, the displayed sets do not overlap for either group. Only queries with two or more of these documents are eligible. Thus, each row shows the same number of documents, and an empty cell cannot show the group. The notebook prints the number of excluded queries. **Consequence:** for controls, raters judge relevant documents that the model did *not* retrieve. This is the same type of document that they judge for failures. The comparison is about the quality of the judgments, not about the retrieved hits.

**Remaining risk and protocol.** The query text is in both sheets. Thus, a rater who sees both sheets can link them. Assign different raters to A and B. Alternatively, the same raters complete A first and B a minimum of one week later. An assertion in the next cell makes sure that the retrieved snippets of a query do not overlap its displayed judged snippets (by document ID and by text).

**Sample.** A maximum of 40 universal failures and 20 controls, selected at random (seed 0) from the eligible queries. **Why these sizes:** a proportion from `n` ratings has a Wilson 95% half-width of approximately `0.98 / sqrt(n)` or less: 0.155 at n = 40, 0.22 at n = 20, 0.10 at n = 96. With 40 failures, a rate of approximately 50% has an error of approximately ±15 points. This is sufficient to tell "mostly" from "rarely", but not to find small differences.

**Rubric (two raters for each sheet, independently):**
- Sheet A, `top_doc_answers_query` (0/1): does one of the three retrieved documents answer or mostly address the query?
- Sheet B, `qrel_doc_plausible` (0/1/2): 0 = the judged-relevant documents do not plausibly relate to the query. 1 = they relate only partly, or only through a broad topic. 2 = they are directly relevant.
- `notes`: free text (for example, "query is a video title", "query is ambiguous").

**Planned analysis** (it runs when the rating columns are filled):
1. Cohen's kappa (with a bootstrap CI) between raters for each sheet.
2. On the consensus: the share of universal failures with a retrieved document that answers the query (a qrels false negative), against controls, with Fisher's exact test.
3. The distribution of `qrel_doc_plausible` by group.

**Output.** With `SAVE_AUDIT_SHEET = True`, the notebook writes the sheets and keys to `audit_out/`. The keys contain qrel data. Do not commit them.

In [ ]:
def snippet(doc_id, n=350):
    return corpus[doc_id][:n].replace("\n", " ")


def displayable_judged(i):
    """Return the judged-relevant docs outside the baseline top 10 whose text matches no retrieved top-3 snippet. Highest level first."""
    top10 = set(base["ids"][i][:K])
    shown_retrieved = {snippet(d) for d in base["ids"][i][:3]}
    cand = sorted(((v, d) for d, v in qrels[qids[i]].items() if v > 0 and d not in top10 and d in corpus), key=lambda x: (-x[0], x[1]))
    return [d for _, d in cand if snippet(d) not in shown_retrieved]


eligible = np.array([len(displayable_judged(i)) >= 2 for i in range(len(qids))])
print("eligible (>= 2 displayable judged docs): universal", int((U & eligible).sum()), "of", int(U.sum()),
      "| controls", int((C & eligible).sum()), "of", int(C.sum()))

rng = np.random.default_rng(SEED)
pick_u = rng.choice(np.where(U & eligible)[0], min(40, int((U & eligible).sum())), replace=False)
pick_c = rng.choice(np.where(C & eligible)[0], min(20, int((C & eligible).sum())), replace=False)
picks = np.r_[pick_u, pick_c]
order_a = np.random.default_rng([SEED, 1]).permutation(len(picks))   # Independent shuffles. Thus, row order and IDs do not link A and B.
order_b = np.random.default_rng([SEED, 2]).permutation(len(picks))

rows_a, key_a, rows_b, key_b = [], [], [], []
for blind, t in enumerate(order_a, 1):
    i = picks[t]
    row = {"blind_id_a": blind, "query_text": queries[qids[i]]}
    for j, d in enumerate(base["ids"][i][:3], 1):
        row[f"retrieved_{j}"] = snippet(d)
    row.update({"rater1_top_doc_answers_query": "", "rater1_notes": "", "rater2_top_doc_answers_query": "", "rater2_notes": ""})
    rows_a.append(row)
    key_a.append({"blind_id_a": blind, "query_id": qids[i], "group": "universal_failure" if U[i] else "control", "split": split[i],
                  "shown_doc_ids": list(base["ids"][i][:3])})
for blind, t in enumerate(order_b, 1):
    i = picks[t]
    shown = displayable_judged(i)[:2]
    row = {"blind_id_b": blind, "query_text": queries[qids[i]]}
    for j, d in enumerate(shown, 1):
        row[f"judged_relevant_{j}"] = snippet(d)
    row.update({"rater1_qrel_doc_plausible": "", "rater1_notes": "", "rater2_qrel_doc_plausible": "", "rater2_notes": ""})
    rows_b.append(row)
    key_b.append({"blind_id_b": blind, "query_id": qids[i], "group": "universal_failure" if U[i] else "control", "split": split[i],
                  "max_qrel_level": int(max_level[i]), "shown_doc_ids": shown})
sheet_a, keyf_a, sheet_b, keyf_b = pd.DataFrame(rows_a), pd.DataFrame(key_a), pd.DataFrame(rows_b), pd.DataFrame(key_b)

# Leak check: link the two sheets through their keys. No query can show the same document (by ID or by text) in both sheets.
link = keyf_a.merge(keyf_b, on=["query_id", "group"], suffixes=("_a", "_b"))
link["id_overlap"] = [len(set(a) & set(b)) for a, b in zip(link["shown_doc_ids_a"], link["shown_doc_ids_b"])]
link["text_overlap"] = [len({snippet(d) for d in a} & {snippet(d) for d in b}) for a, b in zip(link["shown_doc_ids_a"], link["shown_doc_ids_b"])]
link["n_judged_shown"] = link["shown_doc_ids_b"].map(len)
display(link.groupby("group")[["id_overlap", "text_overlap", "n_judged_shown"]].agg(["sum", "min", "max"]))
assert len(link) == len(picks), "every sampled query must appear once in each sheet"
assert link["id_overlap"].sum() == 0 and link["text_overlap"].sum() == 0, "audit leak: a retrieved snippet reappears as a judged document"
assert (link["n_judged_shown"] == 2).all(), "every sheet-B row must show the same number of judged documents"
assert not (keyf_a["query_id"].to_numpy() == keyf_b["query_id"].to_numpy()).all(), "sheets must be shuffled independently"
print("leak check passed: 0 shared documents across", len(link), "queries; blind IDs shuffled independently")

if SAVE_AUDIT_SHEET:
    os.makedirs(AUDIT_DIR, exist_ok=True)
    sheet_a.to_csv(os.path.join(AUDIT_DIR, "audit_sheet_A_retrieval.csv"), index=False)
    sheet_b.to_csv(os.path.join(AUDIT_DIR, "audit_sheet_B_judgments.csv"), index=False)
    keyf_a.to_csv(os.path.join(AUDIT_DIR, "audit_key_A.csv"), index=False)
    keyf_b.to_csv(os.path.join(AUDIT_DIR, "audit_key_B.csv"), index=False)
print(len(picks), "queries sampled:", keyf_a["group"].value_counts().to_dict())
sheet_a[["blind_id_a", "query_text", "retrieved_1"]].head(3)

In [ ]:
# How frequently a single-sheet design leaks the group: a retrieved top-3 snippet appears again among the two
# displayed judged documents (highest level first, ties by document ID). Over all universal failures and controls.
def single_sheet_leak(i):
    shown = [d for _, d in sorted(((v, d) for d, v in qrels[qids[i]].items() if v > 0), key=lambda x: (-x[0], x[1]))[:2]]
    return bool({snippet(d) for d in base["ids"][i][:3]} & {snippet(d) for d in shown if d in corpus})


single = pd.DataFrame({"group": np.where(U, "universal_failure", np.where(C, "control", "other")),
                       "single_sheet_leaks": [single_sheet_leak(i) for i in range(len(qids))]})
single[single["group"] != "other"].groupby("group")["single_sheet_leaks"].agg(["sum", "count", "mean"]).round(3)

In [ ]:
def _kappa_ci(s, a, b, rng):
    kap = cohen_kappa_score(s[a], s[b])
    boot = [cohen_kappa_score(s[a].iloc[i], s[b].iloc[i]) for i in (rng.integers(0, len(s), len(s)) for _ in range(1000))]
    return kap, np.percentile(boot, 2.5), np.percentile(boot, 97.5)


def analyse_ratings(sheet_a_df, key_a_df, sheet_b_df, key_b_df):
    cols_a = ["rater1_top_doc_answers_query", "rater2_top_doc_answers_query"]
    cols_b = ["rater1_qrel_doc_plausible", "rater2_qrel_doc_plausible"]
    for s, cols in ((sheet_a_df, cols_a), (sheet_b_df, cols_b)):
        if (s[cols].astype(str) == "").any().any() or s[cols].isna().any().any():
            print("rating columns are not filled in yet: awaiting two independent raters per sheet. Nothing to analyse.")
            return None
    a, b = sheet_a_df.copy(), sheet_b_df.copy()
    a[cols_a], b[cols_b] = a[cols_a].astype(int), b[cols_b].astype(int)
    rng = np.random.default_rng(SEED)
    print("Cohen's kappa, sheet A (95% bootstrap CI):", tuple(round(x, 3) for x in _kappa_ci(a, *cols_a, rng)))
    print("Cohen's kappa, sheet B (95% bootstrap CI):", tuple(round(x, 3) for x in _kappa_ci(b, *cols_b, rng)))
    ma = a.merge(key_a_df, on="blind_id_a")
    ma["answers"] = ((ma[cols_a[0]] + ma[cols_a[1]]) >= 1).astype(int)   # Either rater: a conservative count
    g = ma.groupby("group")["answers"].agg(["sum", "count"])
    print(g)
    tbl = [[g.loc["universal_failure", "sum"], g.loc["universal_failure", "count"] - g.loc["universal_failure", "sum"]],
           [g.loc["control", "sum"], g.loc["control", "count"] - g.loc["control", "sum"]]]
    print("Fisher exact p (retrieved document answers the query, universal vs control):", stats.fisher_exact(tbl)[1])
    mb = b.merge(key_b_df, on="blind_id_b")
    mb["plausible_mean"] = mb[cols_b].mean(1)
    print(mb.groupby("group")["plausible_mean"].describe())
    return ma, mb


# When two raters fill in the rating columns of each sheet (or of the CSVs in audit_out/, loaded with pd.read_csv), run:
audit_result = analyse_ratings(sheet_a, keyf_a, sheet_b, keyf_b)

In [ ]:
import datasets, matplotlib, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "max_seq_length": MAX_SEQ})

## Findings
All numbers come from one full run (seed 0, NFCorpus, four small encoders, 3,237 queries). A query fails when none of its judged-relevant documents is in the top 10 (recall@10 = 0).

- **Failures are shared across models.** 669 queries fail under all four models. If each model failed independently at its own rate, the expected number is 28 (Monte Carlo p < 0.001, limit 0.0002). Pairwise kappa is 0.70 to 0.77. The failure counts for each model are 1,070 (baseline), 1,004, 921, and 959. 1,918 queries fail under no model. The independence null ignores query difficulty. Thus, the rejection is expected, and it only gives a scale.
- **The largest difference is the number of judged-relevant documents.** Universal failures have a mean of 9.1 relevant documents, and controls have 59.5 (rank-biserial -0.66). Universal failures also have shorter queries (4.4 against 5.3 tokens, rank-biserial -0.15). Both have a Holm-adjusted p < 0.001 in a family of three. The rank of the first relevant document (85 against 1.8) and the cosine from top-1 to the nearest relevant document (0.35 against 0.87) restate the definition of failure. The notebook reports them only descriptively.
- **The level-1 difference is small, but it stays after adjustment for the number of judged documents.** 96.7% of universal failures have a best judgment of level 1, against 91.4% of controls (difference 5.3 points, CI 3.4 to 7.1; Fisher p < 0.001). Almost all queries in both groups have only level-1 judgments. In a logistic regression of universal failure (against controls), the level-1 odds ratio is 2.74 (95% query-bootstrap CI 1.82 to 4.42) unadjusted, and **2.87 (1.82 to 4.79) adjusted for log(n_rel)**. log(n_rel) has its own odds ratio of 0.38 (0.35 to 0.41) for each log unit. Against all other queries, the adjusted odds ratio is 2.59 (1.76 to 4.35). Thus, the number of judged documents does not explain the level-1 association. But the CIs ignore topic clustering.
- **The rate is stable across splits:** 20.7% (train), 18.8% (validation), 22.0% (test). Chi-square p = 0.60.
- **On all queries, a different model fixes more failures than it causes.** Rescued against newly failing: all-MiniLM 202 against 136, bge-small 279 against 130, e5-small 253 against 142 (exact McNemar, Holm-adjusted p < 0.001 for each). On the 323 test queries, the counts are 18 against 16, 26 against 19, and 21 against 17 (Holm-adjusted p = 1.0 for each). The test split alone is too small to show the difference.
- **A single-sheet audit design leaks the group** for 25.9% of controls (496 of 1,918), because a retrieved snippet appears again as a judged document. The two-sheet design (40 universal failures, 20 controls) passes an executed zero-overlap check. No human ratings exist.

### What this supports
- Approximately one in five NFCorpus queries (669 of 3,237) has recall@10 = 0 under all four encoders. These four encoders are small and similar, and their failures are correlated (kappa 0.70 to 0.77). The rate is similar in each split.
- Universal failures have many fewer judged-relevant documents than controls. This is the strongest difference in the data.
- Universal failures more frequently have only level-1 (indirect) judgments, and this association stays after adjustment for the number of judged documents. But the difference in share is small (96.7% against 91.4%).
- On the clean baseline, 1,070 of 3,237 queries (33%) already have recall@10 = 0. Their nDCG cannot decrease. Thus, measure an injected-failure effect only on the queries that can change, and report severity relative to those queries.

### What this does not support
- That these are retrieval failures and not missing relevance judgments. Nothing here separates the two. Only the human audit can.
- That the models share failures "more than chance" in a specific sense. The independence null ignores the large difference in difficulty between queries, so its rejection is trivial. This notebook does not calculate a difficulty-adjusted null.
- That the rank of the first relevant document and the top-1 cosine *explain* failure. They restate it.
- That indirect judgments *cause* universal failure. The adjusted association is observational, its CIs treat queries as independent, and the notebook does not model the query type (video title against question).
- That the audit sample represents all universal failures. 134 of 669 universal failures have fewer than two judged documents outside the top 10, and they cannot enter the blind design.
- That a 20% floor exists for stronger models, other datasets, or other judgment pools. The four encoders are similar in size and training, and their errors are correlated.